# 🚀 LLaMA 7B Fine-tuning for Sharekhan Broking Domain

This notebook fine-tunes LLaMA 2 7B using Unsloth on your custom training data.

**Supports multiple file formats:**
- 📄 DOCX (Word documents)
- 📊 Excel (XLSX/XLS)
- 📑 PDF documents
- 📽️ PowerPoint (PPTX)
- 📝 TXT files
- 📋 JSON (pre-formatted training data)

**Before running:**
1. Go to `Runtime` → `Change runtime type`
2. Select `T4 GPU` as Hardware accelerator
3. Click `Save`

---

## Step 1: Install Dependencies
This will take 2-3 minutes

In [ ]:
%%capture
# Install Unsloth and training dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

# Install document processing dependencies
!pip install python-docx openpyxl pandas PyMuPDF python-pptx

# Disable wandb
import os
os.environ["WANDB_DISABLED"] = "true"

print("✅ Installation complete!")

## Step 2: Check GPU

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 3: Load Base Model
Loading LLaMA 2 7B with 4-bit quantization

In [ ]:
from unsloth import FastLanguageModel

# Configuration
max_seq_length = 2048
dtype = None  # Auto-detect
load_in_4bit = True  # Use 4-bit quantization

# Load model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-2-7b-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print("✅ Model loaded successfully!")

## Step 4: Add LoRA Adapters
Adding trainable adapters to the model

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("✅ LoRA adapters added!")

## Step 5: Document Converter Setup
This converter handles DOCX, Excel, PDF, PPT, and TXT files

In [ ]:
import json
import re
from pathlib import Path
from typing import List, Dict, Tuple

class DocumentConverter:
    """Converts various document formats to training data JSON."""
    
    def __init__(self):
        self.supported_formats = ['.docx', '.xlsx', '.xls', '.pdf', '.pptx', '.ppt', '.txt', '.json']
    
    def convert_file(self, file_path: str, strategy: str = "auto") -> List[Dict]:
        """Convert a file to training data format."""
        ext = Path(file_path).suffix.lower()
        
        if ext == '.json':
            return self._load_json(file_path)
        elif ext == '.docx':
            return self._convert_docx(file_path, strategy)
        elif ext in ['.xlsx', '.xls']:
            return self._convert_excel(file_path)
        elif ext == '.pdf':
            return self._convert_pdf(file_path, strategy)
        elif ext in ['.pptx', '.ppt']:
            return self._convert_pptx(file_path)
        elif ext == '.txt':
            return self._convert_txt(file_path, strategy)
        else:
            raise ValueError(f"Unsupported format: {ext}")
    
    def _load_json(self, file_path: str) -> List[Dict]:
        """Load pre-formatted JSON training data."""
        with open(file_path, 'r', encoding='utf-8') as f:
            return json.load(f)
    
    def _convert_docx(self, file_path: str, strategy: str) -> List[Dict]:
        """Convert DOCX file to training data."""
        from docx import Document
        doc = Document(file_path)
        content = []
        
        for para in doc.paragraphs:
            if para.text.strip():
                if para.style.name.startswith('Heading'):
                    content.append(('heading', para.text.strip()))
                else:
                    content.append(('text', para.text.strip()))
        
        return self._apply_strategy(content, strategy)
    
    def _convert_excel(self, file_path: str) -> List[Dict]:
        """Convert Excel file to training data."""
        import pandas as pd
        df = pd.read_excel(file_path)
        training_data = []
        columns = df.columns.tolist()
        
        # Try to detect Q&A columns
        q_col = a_col = None
        for col in columns:
            col_lower = col.lower()
            if any(q in col_lower for q in ['question', 'query', 'q', 'instruction']):
                q_col = col
            if any(a in col_lower for a in ['answer', 'response', 'a', 'output']):
                a_col = col
        
        if q_col and a_col:
            for _, row in df.iterrows():
                q = str(row[q_col]).strip()
                a = str(row[a_col]).strip()
                if q and a and q != 'nan' and a != 'nan':
                    training_data.append({"instruction": q, "input": "", "output": a})
        elif len(columns) >= 2:
            for _, row in df.iterrows():
                q = str(row[columns[0]]).strip()
                a = str(row[columns[1]]).strip()
                if q and a and q != 'nan' and a != 'nan':
                    training_data.append({"instruction": q, "input": "", "output": a})
        
        return training_data
    
    def _convert_pdf(self, file_path: str, strategy: str) -> List[Dict]:
        """Convert PDF file to training data."""
        import fitz
        doc = fitz.open(file_path)
        content = []
        
        for page in doc:
            text = page.get_text()
            for para in text.split('\n\n'):
                para = para.strip()
                if para:
                    content.append(('text', para))
        doc.close()
        
        return self._apply_strategy(content, strategy)
    
    def _convert_pptx(self, file_path: str) -> List[Dict]:
        """Convert PowerPoint file to training data."""
        from pptx import Presentation
        prs = Presentation(file_path)
        training_data = []
        
        for slide in prs.slides:
            title = ""
            content = []
            for shape in slide.shapes:
                if hasattr(shape, "text") and shape.text.strip():
                    if not title:
                        title = shape.text.strip()
                    else:
                        content.append(shape.text.strip())
            if title and content:
                training_data.append({"instruction": title, "input": "", "output": "\n".join(content)})
        
        return training_data
    
    def _convert_txt(self, file_path: str, strategy: str) -> List[Dict]:
        """Convert TXT file to training data."""
        with open(file_path, 'r', encoding='utf-8') as f:
            text = f.read()
        
        content = []
        for line in text.split('\n'):
            line = line.strip()
            if line:
                if self._is_heading(line):
                    content.append(('heading', self._clean_heading(line)))
                else:
                    content.append(('text', line))
        
        return self._apply_strategy(content, strategy)
    
    def _apply_strategy(self, content: List[Tuple], strategy: str) -> List[Dict]:
        """Apply extraction strategy."""
        if strategy == "auto":
            strategy = self._detect_strategy(content)
        
        if strategy == "faq":
            return self._extract_faq(content)
        elif strategy == "heading":
            return self._extract_heading_content(content)
        else:
            return self._extract_chunks(content)
    
    def _detect_strategy(self, content: List[Tuple]) -> str:
        """Auto-detect best strategy."""
        text = ' '.join([c[1] for c in content if isinstance(c[1], str)])
        faq_patterns = [r'Q:\s*', r'A:\s*', r'Question:', r'Answer:', r'\?\s*\n']
        faq_score = sum(len(re.findall(p, text, re.I)) for p in faq_patterns)
        heading_count = sum(1 for c in content if c[0] == 'heading')
        
        if faq_score > 5:
            return "faq"
        elif heading_count > 3:
            return "heading"
        return "chunk"
    
    def _extract_faq(self, content: List[Tuple]) -> List[Dict]:
        """Extract FAQ-style Q&A pairs."""
        text = '\n'.join([c[1] for c in content if isinstance(c[1], str)])
        training_data = []
        
        # Pattern: Q: ... A: ...
        for match in re.finditer(r'Q[:\.]?\s*(.+?)\s*A[:\.]?\s*(.+?)(?=Q[:\.]?|$)', text, re.DOTALL | re.I):
            q, a = match.group(1).strip(), match.group(2).strip()
            if q and a:
                training_data.append({"instruction": q, "input": "", "output": a})
        
        # Pattern: Question? followed by answer
        if not training_data:
            lines = text.split('\n')
            i = 0
            while i < len(lines):
                if lines[i].strip().endswith('?'):
                    question = lines[i].strip()
                    answer_lines = []
                    i += 1
                    while i < len(lines) and not lines[i].strip().endswith('?'):
                        if lines[i].strip():
                            answer_lines.append(lines[i].strip())
                        i += 1
                    if answer_lines:
                        training_data.append({"instruction": question, "input": "", "output": ' '.join(answer_lines)})
                else:
                    i += 1
        
        return training_data
    
    def _extract_heading_content(self, content: List[Tuple]) -> List[Dict]:
        """Extract heading-content pairs."""
        training_data = []
        current_heading = None
        current_content = []
        
        for item_type, item_content in content:
            if item_type == 'heading':
                if current_heading and current_content:
                    training_data.append({
                        "instruction": current_heading,
                        "input": "",
                        "output": '\n'.join(current_content)
                    })
                current_heading = item_content
                current_content = []
            else:
                current_content.append(item_content)
        
        if current_heading and current_content:
            training_data.append({"instruction": current_heading, "input": "", "output": '\n'.join(current_content)})
        
        return training_data
    
    def _extract_chunks(self, content: List[Tuple], chunk_size: int = 500) -> List[Dict]:
        """Split content into chunks."""
        text = '\n'.join([c[1] for c in content if isinstance(c[1], str)])
        sentences = re.split(r'(?<=[.!?])\s+', text)
        training_data = []
        current_chunk = []
        current_length = 0
        
        for sentence in sentences:
            if current_length + len(sentence) > chunk_size and current_chunk:
                chunk_text = ' '.join(current_chunk)
                topic = ' '.join(chunk_text.split()[:5]) + "..."
                training_data.append({"instruction": f"Explain: {topic}", "input": "", "output": chunk_text})
                current_chunk = []
                current_length = 0
            current_chunk.append(sentence)
            current_length += len(sentence)
        
        if current_chunk:
            chunk_text = ' '.join(current_chunk)
            topic = ' '.join(chunk_text.split()[:5]) + "..."
            training_data.append({"instruction": f"Explain: {topic}", "input": "", "output": chunk_text})
        
        return training_data
    
    def _is_heading(self, line: str) -> bool:
        """Check if line is a heading."""
        if line.startswith('#'): return True
        if re.match(r'^\d+[\.\)]\s+\w', line): return True
        if line.isupper() and len(line) < 100: return True
        if line.endswith(':') and len(line) < 80: return True
        return False
    
    def _clean_heading(self, line: str) -> str:
        """Clean heading text."""
        line = re.sub(r'^#+\s*', '', line)
        line = re.sub(r'^\d+[\.\)]\s*', '', line)
        return line.rstrip(':').strip()

# Create converter instance
converter = DocumentConverter()
print("✅ Document converter ready!")
print(f"Supported formats: {', '.join(converter.supported_formats)}")

## Step 6: Upload and Convert Training Data
Upload your documents to create training data

In [ ]:
from google.colab import files

print("📤 Upload your training documents...")
print("Supported: DOCX, Excel, PDF, PPT, TXT, JSON")
print()

uploaded = files.upload()

# Convert all uploaded files
all_training_data = []

for filename in uploaded.keys():
    try:
        data = converter.convert_file(filename)
        all_training_data.extend(data)
        print(f"✅ {filename}: {len(data)} training examples")
    except Exception as e:
        print(f"❌ {filename}: Error - {e}")

print(f"\n📊 Total training examples: {len(all_training_data)}")

In [ ]:
# Preview the training data
print("📋 Sample training examples:")
print("=" * 50)

for i, example in enumerate(all_training_data[:3]):
    print(f"\n--- Example {i+1} ---")
    print(f"Instruction: {example['instruction'][:100]}..." if len(example['instruction']) > 100 else f"Instruction: {example['instruction']}")
    print(f"Output: {example['output'][:150]}..." if len(example['output']) > 150 else f"Output: {example['output']}")

print(f"\n... and {len(all_training_data) - 3} more examples")

In [ ]:
# Save the converted training data
training_data = all_training_data

# Alpaca prompt template
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

print(f"✅ Training data ready: {len(training_data)} examples")

In [ ]:
from datasets import Dataset

# Format data with Alpaca template
def format_example(example):
    text = alpaca_prompt.format(
        instruction=example["instruction"],
        input=example["input"],
        output=example["output"],
    )
    return {"text": text}

formatted_data = [format_example(ex) for ex in training_data]
dataset = Dataset.from_list(formatted_data)

print(f"✅ Dataset prepared with {len(dataset)} examples")
print(f"\nSample formatted prompt:\n{dataset[0]['text'][:500]}...")

## Step 7: Train the Model
Training time depends on dataset size

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# Adjust epochs based on dataset size
num_epochs = 3 if len(dataset) > 50 else 5

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        output_dir="./outputs",
        num_train_epochs=num_epochs,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        warmup_steps=5,
        logging_steps=1,
        save_steps=50,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        report_to="none",
    ),
)

print(f"Training with {len(dataset)} examples for {num_epochs} epochs...")
trainer_stats = trainer.train()

print(f"\n✅ Training completed!")
print(f"Training time: {trainer_stats.metrics['train_runtime']:.2f}s")
print(f"Final loss: {trainer_stats.metrics['train_loss']:.4f}")

## Step 8: Test the Model
Test with your own questions!

In [ ]:
# Enable inference mode
FastLanguageModel.for_inference(model)

def ask_model(question: str, context: str = ""):
    """Ask the model a question."""
    prompt = alpaca_prompt.format(
        instruction=question,
        input=context,
        output="",
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=300, temperature=0.7, do_sample=True)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("### Response:")[-1].strip()

print("✅ Model ready for inference!")
print("Use: ask_model('Your question here')")

In [ ]:
# Test with first training example
test_question = training_data[0]['instruction']
print(f"Question: {test_question}")
print("=" * 50)
print("Model Response:")
print(ask_model(test_question))

In [ ]:
# Interactive testing - enter your own questions
# Change the question below and run this cell

my_question = "What is the process to open a trading account?"  # <-- Change this!

print(f"Question: {my_question}")
print("=" * 50)
print("Model Response:")
print(ask_model(my_question))

## Step 9: Save Model & Download
Save and download your fine-tuned model

In [ ]:
import os

# Create output directory
save_path = "./finetuned-llama-7b"
os.makedirs(save_path, exist_ok=True)

# Save LoRA adapters
lora_path = f"{save_path}/lora_adapters"
model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)
print(f"✅ LoRA adapters saved to: {lora_path}")

In [ ]:
# Save to GGUF format (for Ollama/llama.cpp)
print("Converting to GGUF format (this may take a few minutes)...")
model.save_pretrained_gguf(
    save_path,
    tokenizer,
    quantization_method="q4_k_m"
)
print(f"✅ GGUF model saved!")

In [ ]:
# Also save the training data for reference
with open('training_data_used.json', 'w', encoding='utf-8') as f:
    json.dump(training_data, f, indent=2, ensure_ascii=False)
print("✅ Training data saved to training_data_used.json")

In [ ]:
# List saved files
print("\n📁 Saved files:")
for root, dirs, files in os.walk(save_path):
    for file in files:
        filepath = os.path.join(root, file)
        size = os.path.getsize(filepath) / (1024*1024)
        print(f"  {file}: {size:.1f} MB")

In [ ]:
# Zip LoRA adapters for download
import shutil
shutil.make_archive('lora_adapters', 'zip', save_path, 'lora_adapters')
print("✅ Created lora_adapters.zip")

In [ ]:
# Download LoRA adapters
print("📥 Downloading LoRA adapters (~50MB)...")
files.download('lora_adapters.zip')

In [ ]:
# Download GGUF model
import glob
gguf_files = glob.glob(f"{save_path}/*.gguf")
if gguf_files:
    print(f"📥 Downloading GGUF file (~4GB)...")
    files.download(gguf_files[0])
else:
    print("No GGUF file found")

In [ ]:
# Download training data used
files.download('training_data_used.json')
print("✅ Downloaded training_data_used.json")

## ✅ Done!

Your fine-tuned model is ready!

**Downloaded files:**
- `lora_adapters.zip` - LoRA adapters (~50MB)
- `*.gguf` - Quantized model (~4GB) for Ollama
- `training_data_used.json` - Training data for reference

**Using with Ollama:**
```bash
# Create Modelfile
echo 'FROM ./finetuned-llama-7b-q4_k_m.gguf' > Modelfile
echo 'TEMPLATE "### Instruction:\n{{.Prompt}}\n\n### Response:\n"' >> Modelfile

# Create and run
ollama create my-model -f Modelfile
ollama run my-model
```